In [12]:
# ============================================================
# CELL 1 — IMPORTS AND LOAD ANALYSIS
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

import py4dgeo


# ------------------------------------------------------------
# Repository and data paths
# ------------------------------------------------------------

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "jupyter":
    REPO_DIR = CURRENT_DIR.parent
else:
    REPO_DIR = CURRENT_DIR


DATA_DIR = (
    REPO_DIR
    / "kijkduin"
)


ANALYSIS_FILE = (
    DATA_DIR
    / "kijkduin.zip"
)


# ------------------------------------------------------------
# Validate analysis archive
# ------------------------------------------------------------

if not ANALYSIS_FILE.exists():
    raise FileNotFoundError(
        "Analysis archive not found. "
        "Run kalman_01_prepare_analysis.ipynb first."
    )


# ------------------------------------------------------------
# Load prepared analysis
# ------------------------------------------------------------

analysis = (
    py4dgeo.SpatiotemporalAnalysis(
        str(
            ANALYSIS_FILE
        )
    )
)


# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

print(
    "Analysis loaded:"
)

print(
    ANALYSIS_FILE
)


print(
    "\nDistances shape:",
    analysis.distances.shape
)


print(
    "Corepoints:",
    analysis.corepoints.cloud.shape[0]
)

Analysis loaded:
C:\Users\tenbi\py4dgeo-kalmanfiltering\kijkduin\kijkduin.zip

Distances shape: (215550, 159)
[2026-08-23 15:09:41][INFO] Restoring epoch from file 'C:\Users\tenbi\AppData\Local\Temp\tmp_cm3ylq9\corepoints.zip'
Corepoints: 215550


In [13]:
# ============================================================
# CELL 2 — COMMON METHOD CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# Analysis corepoint range
# ------------------------------------------------------------
# User-facing numbering is 1-based.
# The corresponding Python indices are created below.

ANALYSIS_RANGE_START = 15000
ANALYSIS_RANGE_END = 15999


n_corepoints = (
    analysis.corepoints.cloud.shape[0]
)


if (
    ANALYSIS_RANGE_START < 1
    or
    ANALYSIS_RANGE_END < ANALYSIS_RANGE_START
    or
    ANALYSIS_RANGE_END > n_corepoints
):
    raise ValueError(
        "Selected analysis corepoint range is invalid."
    )


analysis_indices = list(
    range(
        ANALYSIS_RANGE_START - 1,
        ANALYSIS_RANGE_END
    )
)


# ------------------------------------------------------------
# Standard py4dgeo region-growing parameters
# ------------------------------------------------------------

REGION_GROWING_PARAMETERS = {
    "window_width": 14,
    "minperiod": 2,
    "height_threshold": 0.05,
    "neighborhood_radius": 1.0,
    "min_segments": 10,
    "thresholds": [
        0.3,
        0.4,
        0.5,
        0.6,
        0.7,
        0.8,
        0.9,
    ],
}


# ------------------------------------------------------------
# Kalman seed-detection parameters
# ------------------------------------------------------------

KALMAN_PARAMETERS = {
    "process_sigma": 0.01,
    "min_sigma_obs": 0.005,
    "z_threshold": 1.96,
    "min_seed_duration": 10,
    "min_seed_magnitude": 0.05,
    "seed_direction": "both",
}


# ------------------------------------------------------------
# Configuration summary
# ------------------------------------------------------------

print(
    "Selected analysis corepoints:",
    len(
        analysis_indices
    )
)

print(
    "Corepoint range:",
    f"{ANALYSIS_RANGE_START}–{ANALYSIS_RANGE_END}"
)

Selected analysis corepoints: 1000
Corepoint range: 15000–15999


In [14]:
# ============================================================
# CELL 3 — ORIGINAL 4DOBC BASELINE
# ============================================================

analysis.invalidate_results(
    seeds=True,
    objects=True,
    smoothed_distances=False
)


four_dobc = (
    py4dgeo.RegionGrowingAlgorithm(
        seed_candidates=analysis_indices,
        **REGION_GROWING_PARAMETERS,
    )
)


objects_4dobc = (
    four_dobc.run(
        analysis,
        force=True
    )
)


seeds_4dobc = list(
    analysis.seeds
)


print(
    "Original 4DOBC seeds:",
    len(
        seeds_4dobc
    )
)


print(
    "Original 4DOBC objects:",
    len(
        objects_4dobc
    )
)

[2026-08-23 15:14:52][INFO] Removing intermediate results from the analysis file C:\Users\tenbi\py4dgeo-kalmanfiltering\kijkduin\kijkduin.zip
[2026-08-23 15:14:52][INFO] Removing intermediate results from the analysis file C:\Users\tenbi\py4dgeo-kalmanfiltering\kijkduin\kijkduin.zip
[2026-08-23 15:14:52][INFO] Starting: Find seed candidates in time series
[2026-08-23 15:14:57][INFO] Finished in 4.4768s: Find seed candidates in time series
[2026-08-23 15:14:57][INFO] Starting: Sort seed candidates by priority
[2026-08-23 15:14:58][INFO] Finished in 1.0121s: Sort seed candidates by priority
[2026-08-23 15:14:58][INFO] Starting: Performing region growing on seed candidate 1/2348
[2026-08-23 15:14:58][INFO] Finished in 0.0628s: Performing region growing on seed candidate 1/2348
[2026-08-23 15:14:58][INFO] Starting: Performing region growing on seed candidate 2/2348
[2026-08-23 15:14:58][INFO] Finished in 0.0743s: Performing region growing on seed candidate 2/2348
[2026-08-23 15:14:58][INFO

In [15]:
# ============================================================
# CELL 4 — KF-MAG
# ============================================================

kf_mag = (
    py4dgeo.KalmanRegionGrowingAlgorithm(
        detection_mode="magnitude",
        kalman_corepoint_indices=analysis_indices,

        use_spatial_support=False,
        use_nms=False,

        **KALMAN_PARAMETERS,
        **REGION_GROWING_PARAMETERS,
    )
)


objects_kf_mag = (
    kf_mag.run(
        analysis,
        force=True
    )
)


seeds_kf_mag = list(
    analysis.seeds
)


print(
    "KF-Mag seeds:",
    len(
        seeds_kf_mag
    )
)


print(
    "KF-Mag objects:",
    len(
        objects_kf_mag
    )
)

[2026-08-23 15:17:12][INFO] Removing intermediate results from the analysis file C:\Users\tenbi\py4dgeo-kalmanfiltering\kijkduin\kijkduin.zip
[2026-08-23 15:17:12][INFO] Starting: Find seed candidates in time series
[2026-08-23 15:17:58][INFO] Restoring epoch from file 'C:\Users\tenbi\AppData\Local\Temp\tmpa6z_uqkt\reference_epoch.zip'
[2026-08-23 15:18:00][INFO] Finished in 48.5855s: Find seed candidates in time series
[2026-08-23 15:18:00][INFO] Starting: Sort seed candidates by priority
[2026-08-23 15:18:05][INFO] Finished in 4.3284s: Sort seed candidates by priority
[2026-08-23 15:18:06][INFO] Starting: Performing region growing on seed candidate 1/3117
[2026-08-23 15:18:06][INFO] Finished in 0.1957s: Performing region growing on seed candidate 1/3117
[2026-08-23 15:18:06][INFO] Starting: Performing region growing on seed candidate 3/3117
[2026-08-23 15:18:06][INFO] Finished in 0.1001s: Performing region growing on seed candidate 3/3117
[2026-08-23 15:18:07][INFO] Starting: Perform

In [16]:
# ============================================================
# CELL 5 — KF-RATE
# ============================================================

kf_rate = (
    py4dgeo.KalmanRegionGrowingAlgorithm(
        detection_mode="rate",
        kalman_corepoint_indices=analysis_indices,

        use_spatial_support=False,
        use_nms=False,

        **KALMAN_PARAMETERS,
        **REGION_GROWING_PARAMETERS,
    )
)


objects_kf_rate = (
    kf_rate.run(
        analysis,
        force=True
    )
)


seeds_kf_rate = list(
    analysis.seeds
)


print(
    "KF-Rate seeds:",
    len(
        seeds_kf_rate
    )
)


print(
    "KF-Rate objects:",
    len(
        objects_kf_rate
    )
)

[2026-08-23 15:23:41][INFO] Removing intermediate results from the analysis file C:\Users\tenbi\py4dgeo-kalmanfiltering\kijkduin\kijkduin.zip
[2026-08-23 15:23:41][INFO] Starting: Find seed candidates in time series
[2026-08-23 15:24:18][INFO] Finished in 36.8280s: Find seed candidates in time series
[2026-08-23 15:24:18][INFO] Starting: Sort seed candidates by priority
[2026-08-23 15:24:21][INFO] Finished in 3.3365s: Sort seed candidates by priority
[2026-08-23 15:24:21][INFO] Starting: Performing region growing on seed candidate 1/3719
[2026-08-23 15:24:21][INFO] Finished in 0.1131s: Performing region growing on seed candidate 1/3719
[2026-08-23 15:24:21][INFO] Starting: Performing region growing on seed candidate 3/3719
[2026-08-23 15:24:21][INFO] Finished in 0.0275s: Performing region growing on seed candidate 3/3719
[2026-08-23 15:24:21][INFO] Starting: Performing region growing on seed candidate 20/3719
[2026-08-23 15:24:21][INFO] Finished in 0.0478s: Performing region growing on

In [17]:
# ============================================================
# CELL 6 — KF-MAG-NMS
# ============================================================

kf_mag_nms = (
    py4dgeo.KalmanRegionGrowingAlgorithm(
        detection_mode="magnitude",
        kalman_corepoint_indices=analysis_indices,

        use_spatial_support=True,
        use_nms=True,

        **KALMAN_PARAMETERS,
        **REGION_GROWING_PARAMETERS,
    )
)


objects_kf_mag_nms = (
    kf_mag_nms.run(
        analysis,
        force=True
    )
)


seeds_kf_mag_nms = list(
    analysis.seeds
)


print(
    "KF-Mag-NMS seeds:",
    len(
        seeds_kf_mag_nms
    )
)


print(
    "KF-Mag-NMS objects:",
    len(
        objects_kf_mag_nms
    )
)

[2026-08-23 15:27:51][INFO] Removing intermediate results from the analysis file C:\Users\tenbi\py4dgeo-kalmanfiltering\kijkduin\kijkduin.zip
[2026-08-23 15:27:51][INFO] Starting: Find seed candidates in time series
[2026-08-23 15:28:54][INFO] Finished in 62.7956s: Find seed candidates in time series
[2026-08-23 15:28:54][INFO] Starting: Sort seed candidates by priority
[2026-08-23 15:28:56][INFO] Finished in 2.2186s: Sort seed candidates by priority
[2026-08-23 15:28:56][INFO] Starting: Performing region growing on seed candidate 1/101
[2026-08-23 15:28:56][INFO] Finished in 0.1711s: Performing region growing on seed candidate 1/101
[2026-08-23 15:28:56][INFO] Starting: Performing region growing on seed candidate 2/101
[2026-08-23 15:28:57][INFO] Finished in 0.2788s: Performing region growing on seed candidate 2/101
[2026-08-23 15:28:57][INFO] Starting: Performing region growing on seed candidate 26/101
[2026-08-23 15:28:57][INFO] Finished in 0.1448s: Performing region growing on seed

In [18]:
# ============================================================
# CELL 7 — KF-RATE-NMS
# ============================================================

kf_rate_nms = (
    py4dgeo.KalmanRegionGrowingAlgorithm(
        detection_mode="rate",
        kalman_corepoint_indices=analysis_indices,

        use_spatial_support=True,
        use_nms=True,

        **KALMAN_PARAMETERS,
        **REGION_GROWING_PARAMETERS,
    )
)


objects_kf_rate_nms = (
    kf_rate_nms.run(
        analysis,
        force=True
    )
)


seeds_kf_rate_nms = list(
    analysis.seeds
)


print(
    "KF-Rate-NMS seeds:",
    len(
        seeds_kf_rate_nms
    )
)


print(
    "KF-Rate-NMS objects:",
    len(
        objects_kf_rate_nms
    )
)

[2026-08-23 15:30:08][INFO] Removing intermediate results from the analysis file C:\Users\tenbi\py4dgeo-kalmanfiltering\kijkduin\kijkduin.zip
[2026-08-23 15:30:08][INFO] Starting: Find seed candidates in time series
[2026-08-23 15:31:23][INFO] Finished in 74.6782s: Find seed candidates in time series
[2026-08-23 15:31:23][INFO] Starting: Sort seed candidates by priority
[2026-08-23 15:31:25][INFO] Finished in 2.2467s: Sort seed candidates by priority
[2026-08-23 15:31:25][INFO] Starting: Performing region growing on seed candidate 1/128
[2026-08-23 15:31:25][INFO] Finished in 0.1170s: Performing region growing on seed candidate 1/128
[2026-08-23 15:31:25][INFO] Starting: Performing region growing on seed candidate 2/128
[2026-08-23 15:31:25][INFO] Finished in 0.0753s: Performing region growing on seed candidate 2/128
[2026-08-23 15:31:25][INFO] Starting: Performing region growing on seed candidate 6/128
[2026-08-23 15:31:25][INFO] Finished in 0.0290s: Performing region growing on seed 

In [19]:
# ============================================================
# CELL 8 — METHOD SUMMARY
# ============================================================

summary = pd.DataFrame(
    [
        {
            "method": "4DOBC",
            "n_seeds": len(
                seeds_4dobc
            ),
            "n_objects": len(
                objects_4dobc
            ),
        },

        {
            "method": "KF-Mag",
            "n_seeds": len(
                seeds_kf_mag
            ),
            "n_objects": len(
                objects_kf_mag
            ),
        },

        {
            "method": "KF-Rate",
            "n_seeds": len(
                seeds_kf_rate
            ),
            "n_objects": len(
                objects_kf_rate
            ),
        },

        {
            "method": "KF-Mag-NMS",
            "n_seeds": len(
                seeds_kf_mag_nms
            ),
            "n_objects": len(
                objects_kf_mag_nms
            ),
        },

        {
            "method": "KF-Rate-NMS",
            "n_seeds": len(
                seeds_kf_rate_nms
            ),
            "n_objects": len(
                objects_kf_rate_nms
            ),
        },
    ]
)


display(
    summary
)

,method,n_seeds,n_objects
0,4DOBC,2348,17
1,KF-Mag,3117,15
2,KF-Rate,3719,23
3,KF-Mag-NMS,101,8
4,KF-Rate-NMS,128,11


In [20]:
# ============================================================
# CELL 9 — SAVE COMPACT EXTRACTION RESULTS FOR EVALUATION
# ============================================================

import pickle


# ------------------------------------------------------------
# Evaluation output directory
# ------------------------------------------------------------

EVALUATION_DIR = (
    REPO_DIR
    / "results"
    / "kalman_evaluation"
)


EVALUATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Collect extracted objects by method
# ------------------------------------------------------------

method_objects = {
    "4DOBC": objects_4dobc,
    "KF-Mag": objects_kf_mag,
    "KF-Rate": objects_kf_rate,
    "KF-Mag-NMS": objects_kf_mag_nms,
    "KF-Rate-NMS": objects_kf_rate_nms,
}


# ------------------------------------------------------------
# Convert objects to lightweight records
# ------------------------------------------------------------

saved_objects = {}


for method_name, objects in method_objects.items():

    method_records = []


    for object_number, obj in enumerate(
        objects,
        start=1
    ):

        record = {
            "object_number": object_number,

            "indices": list(
                obj.indices
            ),

            "start_epoch": int(
                obj.start_epoch
            ),

            "end_epoch": int(
                obj.end_epoch
            ),

            "duration_epochs": int(
                obj.end_epoch
                - obj.start_epoch
                + 1
            ),

            "threshold": float(
                obj.threshold
            ),

            "seed_corepoint": int(
                obj.seed.index
            ),

            "seed_start_epoch": int(
                obj.seed.start_epoch
            ),

            "seed_end_epoch": int(
                obj.seed.end_epoch
            ),

            "seed_duration_epochs": int(
                obj.seed.end_epoch
                - obj.seed.start_epoch
                + 1
            ),
        }


        method_records.append(
            record
        )


    saved_objects[
        method_name
    ] = method_records


# ------------------------------------------------------------
# Save compact object data
# ------------------------------------------------------------

with open(
    EVALUATION_DIR
    / "extracted_objects.pkl",
    "wb"
) as file:

    pickle.dump(
        saved_objects,
        file
    )


# ------------------------------------------------------------
# Save Kalman seed tables
# ------------------------------------------------------------

kf_mag.seed_table.to_csv(
    EVALUATION_DIR
    / "kf_mag_seeds.csv",
    index=False
)


kf_rate.seed_table.to_csv(
    EVALUATION_DIR
    / "kf_rate_seeds.csv",
    index=False
)


kf_mag_nms.seed_table.to_csv(
    EVALUATION_DIR
    / "kf_mag_nms_seeds.csv",
    index=False
)


kf_rate_nms.seed_table.to_csv(
    EVALUATION_DIR
    / "kf_rate_nms_seeds.csv",
    index=False
)


# ------------------------------------------------------------
# Save method summary
# ------------------------------------------------------------

summary.to_csv(
    EVALUATION_DIR
    / "method_summary.csv",
    index=False
)


# ------------------------------------------------------------
# Completion message
# ------------------------------------------------------------

print(
    "Evaluation results saved to:"
)

print(
    EVALUATION_DIR
)

Evaluation results saved to:
C:\Users\tenbi\py4dgeo-kalmanfiltering\results\kalman_evaluation
